# Continue Folio local training on T4

Choose **Runtime → Change runtime type → T4 GPU**, connect, and run the setup cells. Upload the separately supplied **Folio_Local_v14_Colab_Transfer.zip** when prompted. This notebook imports the local development adapter and restarts the optimizer for T4. It does not claim exact recovery of the Windows/BF16 run or enable Folio AI.

The data remains mostly synthetic development data with eight reviewed public summaries. Evaluate on T4 before further training. Synthetic JSON scores do not establish general factual accuracy.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
%pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.48.1 'safetensors>=0.4.3'


In [ ]:
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Select and connect a T4 GPU'
print(torch.cuda.get_device_name(0), 'FP16 training on T4')


In [ ]:
import base64,hashlib,io,json,zipfile
from pathlib import Path
TOOLS=Path('/content/folio-tools');TOOLS.mkdir(exist_ok=True)
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='dd03ac9d631f1c5883a4e89aa3d7f1a5f636936f52b8e39f8903c8ffc917d07a'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for info in z.infolist():
        destination=(TOOLS/info.filename).resolve()
        assert destination.is_relative_to(TOOLS.resolve())
        destination.parent.mkdir(parents=True,exist_ok=True)
        destination.write_bytes(z.read(info))


In [ ]:
from google.colab import files
upload=files.upload()
assert len(upload)==1, 'Upload the adapter transfer ZIP only'
name,payload=next(iter(upload.items()))
assert hashlib.sha256(payload).hexdigest()=='4ce3a52f72ccb7db8f2cdaf69379a038bae2b276145d8f797e0bc2abe9b0fa97', 'Transfer ZIP checksum mismatch'
IMPORT=Path('/content/folio-import');IMPORT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    manifest=json.loads(z.read('transfer-manifest.json'))
    assert manifest['model']=='Qwen/Qwen3-4B' and manifest['revision']=='1cfa9a7208912126459214e8b04321603b3df60c'
    for member,spec in manifest['files'].items():
        data=z.read(member)
        assert len(data)==spec['bytes'] and hashlib.sha256(data).hexdigest()==spec['sha256']
        destination=(IMPORT/member).resolve()
        assert destination.is_relative_to(IMPORT.resolve())
        destination.parent.mkdir(parents=True,exist_ok=True);destination.write_bytes(data)
print('Verified adapter and data imported; optimizer will restart')


## Session storage

Drive mounting is disabled. Reports/checkpoints are stored only in this Colab runtime. Download results before disconnecting; a runtime reset can erase them.

In [ ]:
from datetime import datetime,timezone
RUN=Path('/content/FolioTraining')/('transfer-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
RUN.mkdir(parents=True,exist_ok=False)
DATA=IMPORT/'imported/data'
PARENT=IMPORT/'imported/adapter'
print('Temporary reports/checkpoints:',RUN)


## Evaluate the imported adapter first

This generates a fresh T4 base comparison and imported-adapter outputs on the saved development validation/test cases. No training updates occur. Review the generated JSON and answers before continuing.

In [ ]:
subprocess.run(['python','-u',str(TOOLS/'train.py'),'--data',str(DATA),'--output',str(RUN/'preflight'),'--parent-adapter',str(PARENT),'--evaluate-only'],check=True)
report=json.loads((RUN/'preflight/experiment-report.json').read_text())
print(json.dumps({k:v for k,v in report.items() if k not in ('losses','optimized_row_ids')},indent=2))


## Separate development comparison on T4

These 24 frozen cases stay outside training. Compare the base and imported adapter on this T4; local GPU results are not reused. Read the actual answers as well as strict scores. These are developer cases, not blind release qualification.

In [ ]:
from huggingface_hub import snapshot_download
BASE=snapshot_download('Qwen/Qwen3-4B',revision='1cfa9a7208912126459214e8b04321603b3df60c',local_files_only=True)
subprocess.run(['python','-u',str(TOOLS/'evaluate_challenge.py'),'--adapter',str(PARENT),'--base-path',BASE,'--training-data',str(DATA),'--fixtures',str(TOOLS/'development-cases'),'--output',str(RUN/'development-preflight')],check=True)
comparison=json.loads((RUN/'development-preflight/report.json').read_text())
print(json.dumps({k:v for k,v in comparison.items() if k!='cases'},indent=2))


## Continue training — run manually after reviewing the T4 results

One pass over the imported data starts a new optimizer. Checkpoints save every 25 steps. In session-storage mode, download them before disconnecting; otherwise they save to Drive. Further qualification needs independently authored prompts and source-disjoint documents; do not train on evaluation labels.

In [ ]:
rows=json.loads((DATA/'manifest.json').read_text())['splits']['train']['rows']
steps=(rows+3)//4
subprocess.run(['python','-u',str(TOOLS/'train.py'),'--data',str(DATA),'--output',str(RUN/'continued'),'--parent-adapter',str(PARENT),'--baseline-from',str(RUN/'preflight'),'--steps',str(steps),'--accumulation','4','--checkpoint-every','25','--learning-rate','0.00002'],check=True)


## Download results before disconnecting

Run this after evaluations, and again after any further training. It exports the current run including checkpoints. Keep the download to recover your work.

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive(str(RUN)+'-results','zip',root_dir=RUN)
files.download(archive)


Save the adapter and experiment reports to continue local checks. The package and notebook have been checked locally; this notebook has not yet executed on Colab. The new T4 comparison is necessary because precision and library versions differ from the local run.